In [1]:
# Proof of Concept: Multi-Agent Layout Engine
#
# This notebook demonstrates the core concepts of the multi-agent layout architecture for CanvasBot.
# We'll show how to use Pydantic to define a structured layout and then use a Python library
# to generate a PDF based on that layout.

from pydantic import BaseModel, Field
from typing import List, Union, Optional
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import json
from langchain_openai import ChatOpenAI


/media/edwardl.campbell/D/code/CanvasBot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Define the Structured Layout with Pydantic
# We'll create Pydantic models to represent the different elements of a print design, such as text and images.

class TextElement(BaseModel):
    type: str = 'text'
    text: str
    font: str = 'Helvetica'
    size: int = 12
    x_pos: int
    y_pos: int

class ImageElement(BaseModel):
    type: str = 'image'
    path: str
    x_pos: int
    y_pos: int
    width: Optional[int] = None
    height: Optional[int] = None

class DesignLayout(BaseModel):
    """A model to represent a print design layout."""
    width: int = Field(default=int(letter[0]), description="The width of the page.")
    height: int = Field(default=int(letter[1]), description="The height of the page.")
    elements: List[Union[TextElement, ImageElement]] = Field(..., description="The list of design elements on the page.")


In [3]:
# 2. Generate Layout from a Prompt using Local LLM
# We'll use our local vLLM server to generate the design layout from a natural language prompt.

# Connect to the local vLLM server
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="none", # Not needed for local server
    model="Qwen/Qwen2.5-3B-Instruct-AWQ",
    temperature=0.0
)

# Get the JSON schema from the Pydantic model
schema = DesignLayout.model_json_schema()

# Create a prompt for the LLM
user_prompt = "Make me a modern flyer for my coffee shop grand opening on Saturday, offering 20% off."

system_prompt = f"""
You are a world-class graphic design layout agent. Your job is to take a user's request and generate a structured JSON object that represents a print-ready design.

The JSON object must conform to the following JSON schema:
{json.dumps(schema, indent=2)}

- The page size is a standard letter size (width: 612, height: 792).
- Place the elements logically on the page. The origin (0,0) is at the bottom-left corner.
- Do not include any image elements for now. Only generate text elements.
- Respond with ONLY the JSON object, without any additional text or explanations.
"""

# Generate the layout
print("Generating layout from prompt...")
response = llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
])

print("LLM Response:")
print(response.content)

# Parse the LLM's response
try:
    layout_json = json.loads(response.content)
    design = DesignLayout(**layout_json)
    print("\nSuccessfully parsed LLM response into DesignLayout object.")
    print(design.model_dump_json(indent=2))
except (json.JSONDecodeError, TypeError) as e:
    print(f"\nError parsing LLM response: {e}")
    design = None


Generating layout from prompt...
LLM Response:
{
  "width": 612,
  "height": 792,
  "elements": [
    {
      "type": "text",
      "text": "Welcome to [Coffee Shop Name]!",
      "font": "Helvetica",
      "size": 24,
      "x_pos": 150,
      "y_pos": 350
    },
    {
      "type": "text",
      "text": "Limited Time Offer!",
      "font": "Helvetica",
      "size": 18,
      "x_pos": 200,
      "y_pos": 300
    },
    {
      "type": "text",
      "text": "20% OFF your first cup!",
      "font": "Helvetica",
      "size": 18,
      "x_pos": 150,
      "y_pos": 320
    },
    {
      "type": "text",
      "text": "Valid Sat-Sun only.",
      "font": "Helvetica",
      "size": 16,
      "x_pos": 200,
      "y_pos": 340
    },
    {
      "type": "text",
      "text": "[Coffee Shop Name] - Where Coffee Meets Art.",
      "font": "Helvetica",
      "size": 18,
      "x_pos": 150,
      "y_pos": 370
    }
  ]
}

Successfully parsed LLM response into DesignLayout object.
{
  "width": 612,

In [4]:
# 3. Render the PDF
# This function takes the Pydantic layout object and generates a PDF.

def render_pdf(design: DesignLayout, output_path: str):
    if design is None:
        print("Cannot render PDF, design object is None.")
        return
    c = canvas.Canvas(output_path, pagesize=(design.width, design.height))
    for element in design.elements:
        if isinstance(element, TextElement):
            c.setFont(element.font, element.size)
            c.drawString(element.x_pos, element.y_pos, element.text)
        elif isinstance(element, ImageElement):
            # In a real implementation, you would handle image loading and drawing
            # For this PoC, we'll just print a placeholder
            print(f"Drawing image {element.path} at ({element.x_pos}, {element.y_pos})")
    c.save()
    print(f"PDF generated at {output_path}")

render_pdf(design, 'flyer_poc.pdf')


PDF generated at flyer_poc.pdf


In [5]:
# Next Steps
#
# This notebook provides a basic framework. The next steps would be to:
# 1. Integrate an LLM: Use a library like LangChain to generate the `DesignLayout` JSON from a natural language prompt. (DONE)
# 2. Create a FastAPI Endpoint: Wrap the PDF generation logic in a FastAPI endpoint that accepts a `DesignLayout` object.
# 3. Build a Multi-Agent System: Create separate agents for copywriting, image generation, and layout, orchestrated by a main agent.
